# Week 13: Diffusion Models

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week13-diffusion.html

**How to use:** open this notebook in the course codespace (folder `notebooks/`) and run the cells from the top (Shift+Enter, or Run All). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need your key as a Codespaces secret or in the `.env` file. For Ollama examples first run `bash scripts/start-ollama.sh` in the terminal. Model-training examples are much faster on a GPU: open this notebook in Google Colab and choose Runtime → Change runtime type → T4 GPU (in Colab add API keys with the 🔑 Secrets panel).

In [ ]:
# Imports that some of the lesson's code fragments assume
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

## 3. Forward Diffusion Process

In [ ]:
def forward_diffusion_sample(x_0, t, device):
    """
    Takes an image and a timestep as input and
    returns the noisy version of it at timestep t
    """
    noise = torch.randn_like(x_0)
    # t may be a batch of timesteps: reshape the coefficients to (batch, 1, 1, 1)
    # so they broadcast over the channel and spatial dimensions of x_0
    sqrt_alphas_cumprod_t = sqrt_alphas_cumprod[t].reshape(-1, 1, 1, 1).to(device)
    sqrt_one_minus_alphas_cumprod_t = sqrt_one_minus_alphas_cumprod[t].reshape(-1, 1, 1, 1).to(device)

    # Forward diffusion formula: mean + std * noise
    return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise, noise

## 4. Reverse Diffusion Process

In [ ]:
def sample(model, n_samples, device, image_size):
    """
    Samples n_samples new images from the model
    """
    model.eval()
    with torch.no_grad():
        # Start from pure noise
        x = torch.randn(n_samples, 3, image_size, image_size).to(device)

        for i in reversed(range(1, timesteps)):
            t = torch.full((n_samples,), i, device=device, dtype=torch.long)

            # Predict noise
            predicted_noise = model(x, t)

            # Get alpha values for this timestep
            alpha = alphas[t][:, None, None, None]
            alpha_hat = alphas_cumprod[t][:, None, None, None]
            beta = betas[t][:, None, None, None]

            # Only add noise if we're not at the last step
            if i > 1:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)

            # Compute the less-noisy sample x_{t-1} (DDPM reverse step, sigma_t = sqrt(beta_t))
            x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise) + torch.sqrt(beta) * noise

    # Rescale to [0, 1] for images
    return x.clamp(-1, 1).add(1).div(2)

## 5. Neural Network Architecture

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act = nn.SiLU()
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x, t):
        h = self.act(self.norm1(self.conv1(x)))
        time_emb = self.act(self.time_mlp(t))
        h = h + time_emb[:, :, None, None]
        h = self.act(self.norm2(self.conv2(h)))
        return h + self.shortcut(x)

In [ ]:
def timestep_embedding(timesteps, dim, max_period=10000):
    """
    Create sinusoidal timestep embeddings.
    """
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding

## 6. Training Diffusion Models

In [ ]:
def train_step(model, x_0, optimizer):
    batch_size = x_0.shape[0]

    # Sample random timesteps
    t = torch.randint(1, timesteps, (batch_size,), device=x_0.device).long()

    # Get noisy samples (and the noise that was added) via the closed-form forward process
    x_t, noise = forward_diffusion_sample(x_0, t, x_0.device)

    # Predict noise
    predicted_noise = model(x_t, t)

    # Calculate loss
    loss = F.mse_loss(predicted_noise, noise)

    # Update model
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

## Run in your browser

**Forward diffusion: noising an image in closed form (NumPy + matplotlib)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = 1000
betas = np.linspace(1e-4, 0.02, T)                 # linear schedule (Ho et al., 2020)
alpha_bar = np.cumprod(1 - betas)

yy, xx = np.mgrid[-1:1:64j, -1:1:64j]
x0 = np.where(xx**2 + yy**2 < 0.5, 1.0, -1.0)      # a white disk on black, values in [-1, 1]
rng = np.random.default_rng(0)
eps = rng.standard_normal(x0.shape)

steps = [0, 50, 150, 300, 600, 999]
fig, axes = plt.subplots(1, len(steps), figsize=(12, 2.4))
for ax, t in zip(axes, steps):
    xt = np.sqrt(alpha_bar[t]) * x0 + np.sqrt(1 - alpha_bar[t]) * eps
    ax.imshow(xt, cmap="gray", vmin=-2.5, vmax=2.5); ax.axis("off")
    ax.set_title(f"t={t}\nsignal {np.sqrt(alpha_bar[t]):.2f}")
plt.tight_layout()
plt.show()
print("alpha_bar at t = 0, 250, 500, 999:", np.round(alpha_bar[[0, 250, 500, 999]], 4))

## Test code for the course environment

**Noise schedules and the closed-form forward sample (numpy only)**

In [ ]:
# 1. Noise schedules and the closed-form forward sample (numpy only)
import numpy as np

T = 1000
betas_linear = np.linspace(1e-4, 0.02, T)
s = 0.008
tt = np.arange(T + 1) / T
f = np.cos((tt + s) / (1 + s) * np.pi / 2) ** 2
alphas_bar_cos = f / f[0]
betas_cosine = np.clip(1 - alphas_bar_cos[1:] / alphas_bar_cos[:-1], 0, 0.999)

def alpha_bar(betas):
    return np.cumprod(1 - betas)

for name, b in [("linear", betas_linear), ("cosine", betas_cosine)]:
    ab = alpha_bar(b)
    print(f"{name:7s} alpha_bar at t=250/500/750: {ab[249]:.3f} {ab[499]:.3f} {ab[749]:.3f}")

# forward sample of a scalar x0 = 1.0 at t = 500 under the linear schedule
ab = alpha_bar(betas_linear)
x0, t = 1.0, 500
eps = np.random.randn()
x_t = np.sqrt(ab[t - 1]) * x0 + np.sqrt(1 - ab[t - 1]) * eps   # ab[0] is alpha_bar_1
print("x_t =", round(x_t, 3))

**Train a tiny DDPM on a 2-D toy dataset and sample from it (PyTorch)**

In [ ]:
# 2. Train a tiny DDPM on a 2-D toy dataset and sample from it (PyTorch)
import math, torch, torch.nn as nn, torch.nn.functional as F

torch.manual_seed(0)
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alphas_cumprod = torch.cumprod(alphas, 0)

def make_data(n=2048):           # two Gaussian blobs
    c = torch.randint(0, 2, (n, 1)).float() * 4 - 2
    return torch.cat([c, torch.zeros(n, 1)], 1) + 0.3 * torch.randn(n, 2)

def timestep_embedding(timesteps, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half).float() / half)
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], -1)

class EpsNet(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + d, 128), nn.SiLU(),
                                 nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
        self.d = d
    def forward(self, x, t):
        return self.net(torch.cat([x, timestep_embedding(t, self.d)], -1))

model = EpsNet(); opt = torch.optim.Adam(model.parameters(), 1e-3)
x0_all = make_data()
for step in range(1500):
    x0 = x0_all[torch.randint(0, len(x0_all), (256,))]
    t = torch.randint(0, T, (256,))
    noise = torch.randn_like(x0)
    ab = alphas_cumprod[t][:, None]
    x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * noise
    loss = F.mse_loss(model(x_t, t), noise)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0: print(f"step {step} loss {loss.item():.4f}")

@torch.no_grad()
def sample(n=500):
    x = torch.randn(n, 2)
    for i in reversed(range(T)):
        t = torch.full((n,), i)
        eps = model(x, t)
        a, ab, b = alphas[i], alphas_cumprod[i], betas[i]
        z = torch.randn_like(x) if i > 0 else 0
        x = (x - (1 - a) / (1 - ab).sqrt() * eps) / a.sqrt() + b.sqrt() * z
    return x

xs = sample()
print("sampled x-means by cluster:", xs[xs[:, 0] < 0, 0].mean().item(), xs[xs[:, 0] > 0, 0].mean().item())

**Sinusoidal timestep embeddings: nearby timesteps are close, distant ones are not**

In [ ]:
# 3. Sinusoidal timestep embeddings: nearby timesteps are close, distant ones are not
import math, torch

def timestep_embedding(timesteps, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half).float() / half)
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], -1)

e = timestep_embedding(torch.tensor([10, 11, 500]), 32)
cos = torch.nn.functional.cosine_similarity
print("sim(10,11) =", round(cos(e[0:1], e[1:2]).item(), 3))
print("sim(10,500)=", round(cos(e[0:1], e[2:3]).item(), 3))

**Classifier-free guidance arithmetic on noise predictions**

In [ ]:
# 4. Classifier-free guidance arithmetic on noise predictions
import torch

eps_uncond = torch.tensor([0.10, -0.20])   # eps_theta(x_t, None, t)
eps_cond   = torch.tensor([0.40,  0.10])   # eps_theta(x_t, c, t)
for w in [0.0, 1.0, 3.0, 7.5]:
    eps_cfg = eps_uncond + w * (eps_cond - eps_uncond)
    print(f"w={w:<4} eps_cfg={eps_cfg.tolist()}")